# 08 — Lemmatization
**Goal:** Reduce words to dictionary base form using vocabulary + morphology.

Lemmatization maps inflected forms to their dictionary base form — the **lemma**: `running` and `ran` → `run`, `mice` → `mouse`. Unlike stemming (Ch. 09), it consults a vocabulary and the word's part of speech, so the result is always a real word. The catch: it needs a POS signal, which is why Ch. 10 sits where it does in the pipeline.

**Why it matters for resumes / ATS:** resumes and job descriptions say the same skill in every inflection — "Developed", "developing", "develops". Lemma-normalizing makes all three collide with the JD keyword `develop`. But the tool is dangerous: lemmatizing proper nouns (`TensorFlow`, `Google`) silently lowercases and corrupts brand names that must match exactly.

## 1. spaCy Lemmatization

`token.lemma_` is the lemmatized form, and it is **POS-aware**: spaCy tags the token first, then looks up the lemma for that tag. That is why the same surface word can have different lemmas in different contexts — and why irregular forms resolve correctly.

**What the code does:** parses single words and prints `lemma_` with the tag that drove it:
- `running` → `run` (VERB) and `ran` → `run` (VERB) — inflection collapsed, regular and irregular alike
- `better` → `well` (ADV) and `best` → `well` (ADV) — irregular, and *not* `good`
- `was` → `be` (AUX), `mice` → `mouse` (NOUN) — irregular auxiliaries and plurals
- `analyses` → `analysis` (NOUN), `programming` → `programming` (NOUN) — an `-ing` noun is already its own base form

**Try it:** the `better`/`best` → `well` lines are the proof this is vocabulary + morphology, not suffix chopping: no rule-based stemmer gets there (compare Ch. 09).

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
for w in ["running", "ran", "better", "best", "was", "mice", "analyses", "programming"]:
    doc = nlp(w)
    print(f"'{w:15s}' -> '{doc[0].lemma_:15s}' ({doc[0].pos_})")

'running        ' -> 'run            ' (VERB)
'ran            ' -> 'run            ' (VERB)
'better         ' -> 'well           ' (ADV)
'best           ' -> 'well           ' (ADV)
'was            ' -> 'be             ' (AUX)
'mice           ' -> 'mouse          ' (NOUN)
'analyses       ' -> 'analysis       ' (NOUN)
'programming    ' -> 'programming    ' (NOUN)


## 2. spaCy vs NLTK

NLTK's `WordNetLemmatizer` needs the POS passed in — the default is noun. If you lemmatize verbs without saying so, they come back untouched. spaCy infers the POS from context automatically, which is the whole difference in practice.

**What the code does:** compares spaCy against NLTK with noun and verb POS on five words:
- `developed`: spaCy `develop`; NLTK(noun) `developed`; NLTK(verb) `develop` — the default noun call does nothing
- `analyses`: spaCy `analysis`; NLTK(verb) `analyse` — WordNet's British spelling
- `better`: spaCy `well`; NLTK returns `better` either way — no adverb rule wired in by default
- `deployment`: unchanged everywhere — already a base form

**Try it:** the `developed` row is the takeaway — `lem.lemmatize(w)` without `pos='v'` silently no-ops on verbs, the most common source of "why is my lemmatization not working" bugs. spaCy's automatic tagging removes the entire class of error.

In [2]:
from nltk.stem import WordNetLemmatizer
lem = WordNetLemmatizer()
words = ["developed", "developing", "analyses", "better", "deployment"]
for w in words:
    doc = nlp(w)
    print(f"'{w}'  spaCy: {doc[0].lemma_:10s}  NLTK(noun): {lem.lemmatize(w):10s}  NLTK(verb): {lem.lemmatize(w, pos='v'):10s}")

'developed'  spaCy: develop     NLTK(noun): developed   NLTK(verb): develop   
'developing'  spaCy: develop     NLTK(noun): developing  NLTK(verb): develop   
'analyses'  spaCy: analysis    NLTK(noun): analysis    NLTK(verb): analyse   
'better'  spaCy: well        NLTK(noun): better      NLTK(verb): better    
'deployment'  spaCy: deployment  NLTK(noun): deployment  NLTK(verb): deployment


## 3. When NOT to Lemmatize

Proper nouns — company names, tech brands, product names — must match *exactly* as written. Lemmatizing them lowercases and reshapes them, so `TensorFlow` becomes `tensorflow` and an exact match against the JD fails. The rule: lemmatize content words, preserve `PROPN`.

**What the code does:** first checks each term's lemma against its lowercased form, flagging any change:
- `Google` → `Google` — flagged CHANGED because the lemma keeps its capital (differs from `google`)
- `TensorFlow` → `tensorflow` — the case is stripped; flagged unchanged only because the lemma equals the lowercased input
- `Developer` / `Engineering` → lowercased as well

Then `smart_lem()` lemmatizes only non-PROPN tokens. In the stored run both pipelines print the same sentence (`Google develop TensorFlow for develop ML model`) — the hazard is invisible on this sample and shows up on the next unseen brand name, which is precisely why the guard is worth having.

In [4]:
for term in ["Google", "TensorFlow", "Developer", "Engineering"]:
    doc = nlp(term)
    changed = term.lower() != doc[0].lemma_
    print(f"'{term}' -> '{doc[0].lemma_}' {'CHANGED' if changed else 'unchanged'}")

# Solution: preserve PROPN
def smart_lem(doc):
    return [t.text if t.pos_ == "PROPN" else t.lemma_ for t in doc]

text = "Google developed TensorFlow for developing ML models"
doc = nlp(text)
print(f"\nStandard: {' '.join(t.lemma_ for t in doc)}")
print(f"Smart:    {' '.join(smart_lem(doc))}")

'Google' -> 'Google' CHANGED
'TensorFlow' -> 'tensorflow' unchanged
'Developer' -> 'developer' unchanged
'Engineering' -> 'engineering' unchanged

Standard: Google develop TensorFlow for develop ML model
Smart:    Google develop TensorFlow for develop ML model


## Key Insight: Always preserve proper nouns (PROPN). Never lemmatize company names or tech brands.

**Lemmatization wins on content words and destroys proper nouns — so preserve `PROPN` and lemmatize the rest.** Inflection collapsing makes `Developed` and `developing` match `develop`; lemmatizing `TensorFlow` makes it match nothing. The pattern from this chapter — `t.lemma_ if t.pos_ != "PROPN" else t.text` — is the one to ship.

Resume keywords split cleanly along the same line: skills and action verbs benefit from lemmas; company names, tools, and product names need their exact surface form. The next chapter, Ch. 09 stemming, solves the same problem with rules instead of vocabulary — faster, cruder, and mostly wrong for this use case.

**Quick reference — lemma behavior**

| Input | Lemma | POS | Note |
|---|---|---|---|
| `running`, `ran` | `run` | VERB | regular + irregular collapsed |
| `better`, `best` | `well` | ADV | irregular; not `good` |
| `was` | `be` | AUX | auxiliary normalized |
| `mice` | `mouse` | NOUN | irregular plural |
| `analyses` | `analysis` | NOUN | Latin plural, singular lemma |
| `programming` | `programming` | NOUN | `-ing` noun is already base |
| `TensorFlow` | `tensorflow` | PROPN | case stripped — preserve instead |

Use this table as a sanity checklist: if your lemmatizer output for any of these rows differs, your pipeline (POS model, vocabulary, or custom rules) has drifted from the baseline this chapter establishes.

**Pitfalls that cost real matching accuracy**

- Forgetting POS: `WordNetLemmatizer` defaults to noun, so verbs pass through unchanged — always pass `pos='v'` or use a tagger-driven lemmatizer like spaCy.
- Lowercasing brands: a lemma of `tensorflow` will never match the JD text `TensorFlow` under exact comparison; route proper nouns around the lemmatizer.
- Lemmatizing acronyms and version strings: `AWS`, `C++`, `Python 3.11` — lemmatization has nothing to add and can only corrupt them; skip tokens whose shape is already canonical.
- Trusting output blindly: the stored outputs in this chapter are ground truth for the installed spaCy version; a model upgrade can change lemmas (especially irregulars), so re-verify the table above after any environment change.

**Next up — Ch. 09 Stemming**

Lemmatization is the accurate but heavier tool: it needs a vocabulary and a POS tag per token. **Stemming** does the same job with pure suffix-stripping rules — no dictionary, no POS — which makes it fast and vocabulary-free but crude: `engineering` → `engin`, `happily` → `happili`. The next chapter runs Porter vs Snowball on the same resume vocabulary to show exactly where the speed comes from and what it costs. After that, Ch. 10 POS tagging explains how the lemmatizer gets the POS signal it depends on.